# `locityper_00_prep_reference` — GRCh38 FASTA + Jellyfish counts

Run **in a VWB JupyterLab app** before
[`locityper_01_run_stream.ipynb`](locityper_01_run_stream.ipynb). Locityper
needs an **uncompressed** reference + `.fai` and a Jellyfish k-mer database
of that same FASTA ([target docs](https://locityper.vercel.app/target)).

AoU v9 short-read CRAMs are listed at

`workspace/vwb-aou-datasets-controlled-v9/v9/wgs/cram/manifest.csv`

(`person_id`, `cram_uri`, `cram_index_uri` under
`gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/`). This notebook
does **not** download CRAMs; the stream WDL reads them in place.

## What this notebook does

1. Find the CRAM manifest (smoke-test `person_id`)
2. Copy GATK [`Homo_sapiens_assembly38.fasta`](https://support.researchallofus.org/hc/en-us/articles/4807740201876-What-reference-are-the-variants-called-against-for-the-genomic-data) + `.fai` (the FASTA AoU srWGS CRAMs are encoded against: GRCh38, no ALTs, plus hs38d1 / EBV / HLA decoys)
3. `samtools faidx` only if the Broad `.fai` did not copy
4. `jellyfish count` (canonical 25-mers, Locityper defaults)
5. Upload FASTA / fai / `.jf` to a workspace bucket

Need ~15 GB free disk and ≥8 GB RAM (16 GB is safer). `gcloud storage cp`
uses `$GOOGLE_CLOUD_PROJECT` as the requester-pays billing project when set.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import os
import shutil
import subprocess
import sys
from pathlib import Path


def sh(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check, text=True)


def capture(cmd: list[str], *, check: bool = True) -> str:
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, check=check, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
    if proc.stderr:
        print(proc.stderr, end="" if proc.stderr.endswith("\n") else "\n", file=sys.stderr)
    return proc.stdout


def md5sum(path: Path, *, buf: int = 1024 * 1024) -> str:
    h = hashlib.md5()
    with path.open("rb") as fh:
        while True:
            chunk = fh.read(buf)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def find_manifest() -> Path | None:
    env = os.environ.get("LOCITYPER_CRAM_MANIFEST", "").strip()
    if env:
        p = Path(env)
        if p.is_file():
            return p.resolve()
    here = Path.cwd().resolve()
    names = [
        Path("workspace/vwb-aou-datasets-controlled-v9/v9/wgs/cram/manifest.csv"),
        Path("v9/wgs/cram/manifest.csv"),
    ]
    roots = [here, *here.parents, Path("/home/jupyter"), Path("/home/jupyter/workspace")]
    for root in roots:
        for rel in names:
            cand = (root / rel) if not rel.is_absolute() else rel
            if cand.is_file():
                return cand.resolve()
    ws = Path("workspace")
    if ws.is_dir():
        hits = list(ws.glob("**/v9/wgs/cram/manifest.csv"))
        if hits:
            return hits[0].resolve()
    return None


SCRATCH = Path.cwd() / "locityper_rw_scratch" / "refs"
SCRATCH.mkdir(parents=True, exist_ok=True)

# GATK GRCh38 used for AoU srWGS CRAMs (no ALTs; hs38d1 / EBV / HLA decoys).
# https://support.researchallofus.org/hc/en-us/articles/4807740201876
REF_STEM = "Homo_sapiens_assembly38"
REF_FA_NAME = f"{REF_STEM}.fasta"
REF_FA_GCS_CANDIDATES = [
    "gs://gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta",
    "gs://genomics-public-data/resources/broad/hg38/v0/Homo_sapiens_assembly38.fasta",
]
REF_FAI_GCS_CANDIDATES = [
    "gs://gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta.fai",
    "gs://genomics-public-data/references/hg38/v0/Homo_sapiens_assembly38.fasta.fai",
]
REF_FA_GCS = os.environ.get("LOCITYPER_REF_FA_GCS", REF_FA_GCS_CANDIDATES[0])
REF_FAI_GCS = os.environ.get("LOCITYPER_REF_FAI_GCS", REF_FAI_GCS_CANDIDATES[0])

JF_NAME = f"counts.{REF_STEM}.k25.jf"
JF_K = 25
JF_SIZE = os.environ.get("LOCITYPER_JF_SIZE", "3G")
JF_THREADS = int(os.environ.get("LOCITYPER_JF_THREADS", str(min(os.cpu_count() or 8, 16))))

LOCITYPER_DOCKER = os.environ.get("LOCITYPER_DOCKER", "eichlerlab/locityper:1.4.5.0")

OUTPUT_BUCKET_ID = os.environ.get("LOCITYPER_OUTPUT_BUCKET_ID", "")
OUTPUT_BUCKET_GS = os.environ.get("LOCITYPER_OUTPUT_BUCKET_GS", "")
STAGE_PREFIX = os.environ.get("LOCITYPER_STAGE_PREFIX", "locityper/refs")

SMOKE_PERSON_ID = os.environ.get("LOCITYPER_SMOKE_PERSON_ID", "1000000")

FORCE_DOWNLOAD = False
FORCE_FAIDX = False
FORCE_JELLYFISH = False
UPLOAD = True
CONDA_INSTALL = True  # if samtools/jellyfish are missing and conda exists

REF_FA = SCRATCH / REF_FA_NAME
REF_FAI = Path(str(REF_FA) + ".fai")
COUNTS_JF = SCRATCH / JF_NAME

print("SCRATCH:", SCRATCH)
print("REF_FA_GCS:", REF_FA_GCS)
print("REF_FAI_GCS:", REF_FAI_GCS)
print("disk free GiB:", round(shutil.disk_usage(SCRATCH).free / 1024**3, 1))
print("JF_THREADS:", JF_THREADS, "JF_SIZE:", JF_SIZE)
print("OUTPUT_BUCKET_ID:", OUTPUT_BUCKET_ID or "(unset)")
print("samtools:", shutil.which("samtools"), "jellyfish:", shutil.which("jellyfish"))
print("docker:", shutil.which("docker"), "conda:", shutil.which("conda") or shutil.which("mamba"))


## Tools

Need `samtools` and `jellyfish` on the VM, or Docker with the Locityper image
(Isaac’s `eichlerlab/locityper:1.4.5.0` already has both). Building the
counts file with that image avoids a Jellyfish 1 vs 2 format mismatch at
genotype time.


In [ ]:
def conda_bin() -> str | None:
    return shutil.which("mamba") or shutil.which("conda")


def ensure_conda_tools() -> None:
    if shutil.which("samtools") and shutil.which("jellyfish"):
        return
    exe = conda_bin()
    if not CONDA_INSTALL or exe is None:
        return
    sh(
        [
            exe,
            "install",
            "-y",
            "-c",
            "conda-forge",
            "-c",
            "bioconda",
            "samtools",
            "jellyfish",
        ]
    )


ensure_conda_tools()


def docker_wrap(tool: str) -> list[str] | None:
    if shutil.which("docker") is None:
        return None
    work = str(SCRATCH.resolve())
    return [
        "docker",
        "run",
        "--rm",
        "-v",
        f"{work}:{work}",
        "-w",
        work,
        LOCITYPER_DOCKER,
        tool,
    ]


def tool_cmd(tool: str) -> list[str]:
    if shutil.which(tool):
        return [tool]
    wrapped = docker_wrap(tool)
    if wrapped:
        print(f"using docker {LOCITYPER_DOCKER} for {tool}")
        return wrapped
    raise SystemExit(
        f"{tool} is not on PATH and docker is unavailable. "
        "Install bioconda samtools+jellyfish, or pull the Locityper image."
    )


SAMTOOLS = tool_cmd("samtools")
JELLYFISH = tool_cmd("jellyfish")
print("SAMTOOLS:", SAMTOOLS)
print("JELLYFISH:", JELLYFISH)
sh(SAMTOOLS + ["--version"], check=False)
sh(JELLYFISH + ["--version"], check=False)


## CRAM manifest

v9 WGS CRAMs (Illumina, `v8_base`). Use a `person_id` from this table as
`SAMPLE_ID` in the run notebook; pass `cram_uri` / `cram_index_uri` as
`CRAM` / `CRAI`.


In [ ]:
MANIFEST = find_manifest()
print("MANIFEST:", MANIFEST or "(not found — set LOCITYPER_CRAM_MANIFEST)")

SMOKE_CRAM = ""
SMOKE_CRAI = ""
if MANIFEST is not None:
    with MANIFEST.open(newline="") as fh:
        rows = list(csv.DictReader(fh))
    print("n samples:", len(rows))
    print("columns:", list(rows[0].keys()) if rows else [])
    hit = next((r for r in rows if r.get("person_id") == str(SMOKE_PERSON_ID)), None)
    if hit is None and rows:
        hit = rows[0]
        SMOKE_PERSON_ID = hit["person_id"]
        print("SMOKE_PERSON_ID not in manifest; using first row", SMOKE_PERSON_ID)
    if hit:
        SMOKE_CRAM = hit["cram_uri"]
        SMOKE_CRAI = hit["cram_index_uri"]
        print("smoke person_id:", SMOKE_PERSON_ID)
        print("cram:", SMOKE_CRAM)
        print("crai:", SMOKE_CRAI)
    print("--- head ---")
    for row in rows[:5]:
        print(row["person_id"], row["cram_uri"])


## Workspace bucket

Uploads go to `OUTPUT_BUCKET_GS / STAGE_PREFIX`. `--output-bucket-id` in the
run notebook is this resource ID, not the `gs://` name.


In [ ]:
if shutil.which("wb"):
    sh(["wb", "auth", "status"], check=False)
    sh(["wb", "resource", "list", "--type=GCS_BUCKET"], check=False)
    if OUTPUT_BUCKET_ID and not OUTPUT_BUCKET_GS:
        resolved = capture(
            ["wb", "resource", "resolve", f"--id={OUTPUT_BUCKET_ID}"],
            check=False,
        ).strip().splitlines()
        resolved = next((line.strip() for line in reversed(resolved) if line.strip()), "")
        if resolved.startswith("gs://") or resolved.startswith("s3://"):
            OUTPUT_BUCKET_GS = resolved
        elif resolved and " " not in resolved:
            OUTPUT_BUCKET_GS = f"gs://{resolved}"
else:
    print("wb not on PATH; set OUTPUT_BUCKET_GS to a gs:// prefix you can write")

print("OUTPUT_BUCKET_GS:", OUTPUT_BUCKET_GS or "(unset)")
if not OUTPUT_BUCKET_GS:
    print("Uploads will be skipped until OUTPUT_BUCKET_ID/GS is set.")


## Copy the FASTA

AoU’s published srWGS reference is already uncompressed. Re-run with
`FORCE_DOWNLOAD=True` to replace a local copy. Override `REF_FA_GCS` /
`REF_FAI_GCS` if those public buckets are not readable from this workspace.


In [ ]:
def gcs_cp(src: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    cmd = ["gcloud", "storage", "cp", src, str(dest)]
    project = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT")
    if project:
        cmd.insert(3, f"--billing-project={project}")
    sh(cmd)


def gcs_cp_first(uris: list[str], dest: Path) -> str:
    last_err: Exception | None = None
    for uri in uris:
        try:
            gcs_cp(uri, dest)
            return uri
        except subprocess.CalledProcessError as exc:
            print("failed", uri)
            last_err = exc
    raise SystemExit(f"could not copy to {dest}: {last_err}") from last_err


need_fa = FORCE_DOWNLOAD or not REF_FA.is_file()
if need_fa:
    REF_FA.unlink(missing_ok=True)
    uris = [REF_FA_GCS] + [u for u in REF_FA_GCS_CANDIDATES if u != REF_FA_GCS]
    used = gcs_cp_first(uris, REF_FA)
    print("copied FASTA from", used)
else:
    print("skip FASTA copy; exists", REF_FA)

print("fasta bytes:", REF_FA.stat().st_size)

need_fai = FORCE_DOWNLOAD or FORCE_FAIDX or not REF_FAI.is_file()
if need_fai:
    REF_FAI.unlink(missing_ok=True)
    uris = [REF_FAI_GCS] + [u for u in REF_FAI_GCS_CANDIDATES if u != REF_FAI_GCS]
    try:
        used = gcs_cp_first(uris, REF_FAI)
        print("copied fai from", used)
    except SystemExit:
        print("no public fai; will samtools faidx in the next cell")
else:
    print("skip fai copy; exists", REF_FAI)


## `faidx`

Skip if the Broad `.fai` copied. The stream WDL still wants the FASTA and
index as separate files (Isaac staged `reference.fa` once on Terra).


In [ ]:
if REF_FAI.is_file() and not FORCE_FAIDX:
    print("skip faidx; exists", REF_FAI)
else:
    sh(SAMTOOLS + ["faidx", str(REF_FA)])

print("--- .fai head ---")
print("\n".join(REF_FAI.read_text().splitlines()[:8]))


## Jellyfish counts

Locityper’s documented command ([target](https://locityper.vercel.app/target)):

```bash
jellyfish count --canonical --lower-count 2 --out-counter-len 2 --mer-len 25     --threads 8 --size 3G --output counts.jf reference.fa
```

Human 25-mers at `--size 3G` is typically 10–30 min. If jellyfish shards the
output (`*.jf_0`, `*.jf_1`), the hash was too small — raise `JF_SIZE` and
re-run with `FORCE_JELLYFISH=True`.


In [ ]:
if FORCE_JELLYFISH or not COUNTS_JF.is_file():
    COUNTS_JF.unlink(missing_ok=True)
    for leftover in SCRATCH.glob(COUNTS_JF.name + "_*"):
        leftover.unlink()
    sh(
        JELLYFISH
        + [
            "count",
            "--canonical",
            "--lower-count",
            "2",
            "--out-counter-len",
            "2",
            "--mer-len",
            str(JF_K),
            "--threads",
            str(JF_THREADS),
            "--size",
            str(JF_SIZE),
            "--output",
            str(COUNTS_JF),
            str(REF_FA),
        ]
    )
else:
    print("skip jellyfish; exists", COUNTS_JF)

shards = sorted(SCRATCH.glob(COUNTS_JF.name + "_*"))
if shards:
    raise SystemExit(
        f"jellyfish overflowed the hash table: {shards}. "
        "Set JF_SIZE to 4G or 6G and FORCE_JELLYFISH=True."
    )
if not COUNTS_JF.is_file():
    raise SystemExit(f"expected {COUNTS_JF}")

print("jf bytes:", COUNTS_JF.stat().st_size)
sh(JELLYFISH + ["stats", str(COUNTS_JF)], check=False)


## Upload and print run-notebook assignments


In [ ]:
REF_FA_GS = ""
REF_FAI_GS = ""
COUNTS_JF_GS = ""

if UPLOAD:
    if not OUTPUT_BUCKET_GS:
        raise SystemExit("Set OUTPUT_BUCKET_ID (and re-run the bucket cell) before uploading.")
    prefix = f"{OUTPUT_BUCKET_GS.rstrip('/')}/{STAGE_PREFIX.strip('/')}"
    REF_FA_GS = f"{prefix}/{REF_FA.name}"
    REF_FAI_GS = f"{prefix}/{REF_FAI.name}"
    COUNTS_JF_GS = f"{prefix}/{COUNTS_JF.name}"
    for src, dest in (
        (REF_FA, REF_FA_GS),
        (REF_FAI, REF_FAI_GS),
        (COUNTS_JF, COUNTS_JF_GS),
    ):
        sh(["gcloud", "storage", "cp", str(src), dest])
else:
    print("UPLOAD=False; files stay on this VM only")
    prefix = "(not uploaded)"

print(
    f"""
Paste into locityper_01_run_stream config:

SAMPLE_ID = "{SMOKE_PERSON_ID}"
CRAM = "{SMOKE_CRAM}"
CRAI = "{SMOKE_CRAI}"
REF_FA = "{REF_FA_GS or REF_FA}"
REF_FAI = "{REF_FAI_GS or REF_FAI}"
COUNTS_JF = "{COUNTS_JF_GS or COUNTS_JF}"
TECHNOLOGY = "illumina"
"""
)
